### Step1: Load Uber Pickups NYC dataset

In [2]:
import pandas as pd
import numpy as np

# Step 1: Load and Preprocess Uber Dataset
file_path = './data/all_uber.csv'  # Download from URL above
data = pd.read_csv(file_path)

# Change to numerical timestamp
data['datetime'] = pd.to_datetime(data['Date/Time'])
min_dt = data['datetime'].min()
max_dt = data['datetime'].max()
# print("min_dt", min_dt, "max_dt", max_dt)
data['timestamp'] = (data['datetime'] - min_dt).dt.total_seconds()
data = data[['timestamp', 'Lat', 'Lon', 'Base']].dropna()  # Ignore Base for now
full_data_size = len(data)
# Dimensions for ranges (3D: time + spatial bbox)
dimensions = ['timestamp', 'Lat', 'Lon']

print(f"Dataset loaded: {full_data_size} rows")
print(data.head())

Dataset loaded: 4534327 rows
   timestamp      Lat      Lon    Base
0      660.0  40.7690 -73.9549  B02512
1     1020.0  40.7267 -74.0345  B02512
2     1260.0  40.7316 -73.9873  B02512
3     1680.0  40.7588 -73.9776  B02512
4     1980.0  40.7594 -73.9722  B02512


### Step2: Create a Small Offline Sample

In [3]:
# Step 2: Create Small Offline Sample
sample_size = int(0.01 * full_data_size)  # ~5k rows
sample = data.sample(n=sample_size, random_state=42).copy()
print(f"Sample created: {sample.shape[0]} rows")
print(sample.head())

Sample created: 45343 rows
          timestamp      Lat      Lon    Base
255777     850500.0  40.7588 -73.9726  B02617
2591794   7138080.0  40.8451 -73.9418  B02617
1384006  12956280.0  40.7399 -73.9823  B02764
530592    2392200.0  40.6449 -73.7824  B02682
3679163  14676000.0  40.7636 -73.9798  B02598


In [4]:
from query_generate import generate_bounded_random_query

# generate a random query to test and see min, max value.
q = generate_bounded_random_query(data, dimensions, test = True)
print(q)

dim: timestamp, min value: 0.0, max value: 15807540.0
dim: Lat, min value: 39.6569, max value: 42.1166
dim: Lon, min value: -74.929, max value: -72.0666
{'timestamp': (3159991.5485581327, 14866508.535251021), 'Lat': (40.24640292284147, 42.11482270005103), 'Lon': (-74.8239112012194, -72.24850144972031)}


Build query log (about 3~4 mins), the main task is COUNT.

In [5]:
from query_generate import generate_uber_query_log

task = "COUNT"
# number of queries
num_queries = 2000

query_log = generate_uber_query_log(num_queries, data, sample, dimensions, full_data_size)

Generating uber query log...
Generated 2000 queries


In [6]:
new_query = generate_bounded_random_query(data, dimensions, test = True)
print(new_query)

dim: timestamp, min value: 0.0, max value: 15807540.0
dim: Lat, min value: 39.6569, max value: 42.1166
dim: Lon, min value: -74.929, max value: -72.0666
{'timestamp': (1878122.6388244638, 12197921.12936671), 'Lat': (40.023988670166986, 41.808968089389936), 'Lon': (-74.46630092845052, -72.42976148293604)}


In [9]:
from query_calculate import sample_count, exact_count


def range_distance(q1, q2, dimensions):
    dist = 0.0
    for dim in dimensions:
        l1, r1 = q1[dim]
        l2, r2 = q2[dim]
        dist += abs(l1 - l2) + abs(r1 - r2)
    return dist

# Find error-similar historical query (closest error)
min_dist = float('inf')
opt_entry = None
for q in query_log:
    dist = range_distance(new_query, q['query'], dimensions)
    if dist < min_dist:
        min_dist = dist
        opt_entry = q

# Compute final estimate
sample_new = sample_count(new_query, sample, full_data_size)
sample_opt = opt_entry['estimate']
final_estimate = opt_entry['exact'] + (sample_new - sample_opt)

print(f"AQP++ estimate: {final_estimate:.2f}")
# Compute exact for the same query (for debugging/small queries)
exact = exact_count(new_query, data)
print(f"Exact count: {exact:.2f}")
print(f"Relative error: {abs(final_estimate - exact) / exact:.4f}")

AQP++ estimate: 2813907.85
Exact count: 2815133.00
Relative error: 0.0004


### Step 6: Evaluate and Extend

In [12]:
evaluation_query_log = generate_uber_query_log(100, data, sample, dimensions, full_data_size)

Generating uber query log...
Generated 100 queries


In [15]:
aqpp_errors = []
aqpp_abs_errors = []
aqpp_sq_errors = []

for entry in evaluation_query_log:
    new_query = entry['query']

    # --- select q_old by range similarity ---
    min_dist = float('inf')
    opt_entry = None
    for q in query_log:
        dist = range_distance(new_query, q['query'], dimensions)
        if dist < min_dist:
            min_dist = dist
            opt_entry = q

    # --- AQP++ estimation ---
    q_old_exact = opt_entry['exact']
    q_hat_new   = sample_count(new_query, sample, full_data_size)
    q_hat_old   = sample_count(opt_entry['query'], sample, full_data_size)
    aqpp_est    = q_old_exact + (q_hat_new - q_hat_old)

    # --- exact ---
    exact = exact_count(new_query, data)

    # skip tiny exact queries (same rule as你原本)
    if exact == 0:
        continue

    # --- error metrics ---
    rel_err = abs(aqpp_est - exact) / exact
    aqpp_errors.append(rel_err)
    aqpp_abs_errors.append(abs(aqpp_est - exact))
    aqpp_sq_errors.append((aqpp_est - exact) ** 2)

aqpp_ARE    = np.mean(aqpp_errors)
aqpp_Median = np.median(aqpp_errors)
aqpp_MSE    = np.mean(aqpp_sq_errors)
aqpp_MAE    = np.mean(aqpp_abs_errors)

print(f"AQP++ Average Relative Error (ARE): {aqpp_ARE:.4f}")
print(f"AQP++ Median Relative Error:        {aqpp_Median:.4f}")
print(f"AQP++ Mean Squared Error (MSE):     {aqpp_MSE:.2f}")
print(f"AQP++ Mean Absolute Error (MAE):    {aqpp_MAE:.2f}")


AQP++ Average Relative Error (ARE): 0.0003
AQP++ Median Relative Error:        0.0002
AQP++ Mean Squared Error (MSE):     1662058.41
AQP++ Mean Absolute Error (MAE):    986.70
